In [4]:
import os
import json
import pandas as pd
import numpy as np

class PADSLoader:
    def __init__(self, base_path: str):
        """
        Initialize the loader with the base dataset directory.
        
        Args:
            base_path (str): Path to the root of PADS dataset
        """
        self.base_path = base_path

    def load_metadata(self, patient_id: int) -> dict:
        """Load patient metadata from patients folder."""
        path = os.path.join(self.base_path, "patients", f"patient_{patient_id:03d}.json")
        with open(path, "r") as f:
            return json.load(f)

    def load_questionnaire(self, patient_id: int) -> dict:
        """Load questionnaire responses."""
        path = os.path.join(self.base_path, "questionnaire", f"questionnaire_response_{patient_id:03d}.json")
        with open(path, "r") as f:
            return json.load(f)

    def load_movement_metadata(self, patient_id: int) -> dict:
        """Load movement observation metadata."""
        path = os.path.join(self.base_path, "movement", f"observation_{patient_id:03d}.json")
        with open(path, "r") as f:
            return json.load(f)

    def load_timeseries(self, patient_id: int) -> dict:
        """
        Load all timeseries .txt files for a patient.
        Returns a dict {filename: DataFrame}.
        """
        folder = os.path.join(self.base_path, "movement", "timeseries")
        timeseries_data = {}

        for file in os.listdir(folder):
            if file.startswith(f"{patient_id:03d}_") and file.endswith(".txt"):
                path = os.path.join(folder, file)
                df = pd.read_csv(path, header=None)
                timeseries_data[file] = df
        
        return timeseries_data

    def load_preprocessed(self, patient_id: int) -> np.ndarray:
        """Load preprocessed binary data (if available)."""
        path = os.path.join(self.base_path, "preprocessed", "movement", f"{patient_id:03d}.bin")
        if os.path.exists(path):
            return np.fromfile(path, dtype=np.float32)
        return None

    def load_all(self, patient_id: int) -> dict:
        """
        Load all modalities for a patient.
        Returns:
            dict with keys: metadata, questionnaire, movement_meta, timeseries, preprocessed
        """
        return {
            "metadata": self.load_metadata(patient_id),
            "questionnaire": self.load_questionnaire(patient_id),
            "movement_meta": self.load_movement_metadata(patient_id),
            "timeseries": self.load_timeseries(patient_id),
            "preprocessed": self.load_preprocessed(patient_id),
        }


In [5]:
loader = PADSLoader("/users/imbahndu/Desktop/Columbia DBM/Smartwatch dataset_In progress")

patient_data = loader.load_all(1)

print(patient_data["metadata"]["age"])
print(patient_data["questionnaire"]["item"][0])
print(patient_data["movement_meta"]["session"][0])
print(patient_data["timeseries"].keys())      # all txt files
print(patient_data["preprocessed"].shape)     # preprocessed data


56
{'link_id': '01', 'text': 'Dribbling of saliva during the daytime', 'answer': False}
{'record_name': 'Relaxed', 'rows': 2048, 'records': [{'device_location': 'LeftWrist', 'channels': ['Time', 'Accelerometer_X', 'Accelerometer_Y', 'Accelerometer_Z', 'Gyroscope_X', 'Gyroscope_Y', 'Gyroscope_Z'], 'units': ['s', 'g', 'g', 'g', 'rad/s', 'rad/s', 'rad/s'], 'file_name': 'timeseries/001_Relaxed_LeftWrist.txt'}, {'device_location': 'RightWrist', 'channels': ['Time', 'Accelerometer_X', 'Accelerometer_Y', 'Accelerometer_Z', 'Gyroscope_X', 'Gyroscope_Y', 'Gyroscope_Z'], 'units': ['s', 'g', 'g', 'g', 'rad/s', 'rad/s', 'rad/s'], 'file_name': 'timeseries/001_Relaxed_RightWrist.txt'}]}
dict_keys(['001_CrossArms_LeftWrist.txt', '001_CrossArms_RightWrist.txt', '001_DrinkGlas_LeftWrist.txt', '001_DrinkGlas_RightWrist.txt', '001_Entrainment_LeftWrist.txt', '001_Entrainment_RightWrist.txt', '001_HoldWeight_LeftWrist.txt', '001_HoldWeight_RightWrist.txt', '001_LiftHold_LeftWrist.txt', '001_LiftHold_Right

AttributeError: 'NoneType' object has no attribute 'shape'

In [6]:
import pandas as pd
import numpy as np
import os

# Example folder and files (adjust as needed)
timeseries_folder = "timeseries"
files = [
    "001_Relaxed_LeftWrist.txt",
    "001_Relaxed_RightWrist.txt",
    # Add more files here
]

# Example label from questionnaire (change to actual label logic)
label = int(patient_data["questionnaire"][56]['answer'])  # 0 or 1

window_size = 100  # samples per window
step_size = 50     # sliding step

def extract_features(df):
    features = {}
    for col in ['Accelerometer_X', 'Accelerometer_Y', 'Accelerometer_Z',
                'Gyroscope_X', 'Gyroscope_Y', 'Gyroscope_Z']:
        features[f"{col}_mean"] = df[col].mean()
        features[f"{col}_std"] = df[col].std()
    return features

rows = []

for file_name in files:
    file_path = os.path.join(timeseries_folder, file_name)
    df = pd.read_csv(file_path, sep='\t')  # or ',' depending on your file
    for start in range(0, len(df) - window_size + 1, step_size):
        window = df.iloc[start:start+window_size]
        feats = extract_features(window)
        feats["label"] = label  # label for this window
        feats["file"] = file_name
        feats["start_index"] = start
        rows.append(feats)

# Create DataFrame and save CSV
features_df = pd.DataFrame(rows)
features_df.to_csv("features_labels.csv", index=False)

print("CSV file 'features_labels.csv' created with shape:", features_df.shape)


KeyError: 56